In [1]:
from pathlib import Path 
from phd_helpers.AbaqusPostprocessing import inp2pv, get_field_path, get_field_df, add_field_to_mesh, get_history_path
import numpy as np
import pandas as pd
import json

In [2]:
def get_step_ids(csv_dir):
    history_files = list(csv_dir.glob('history*.csv'))
    step_ids = np.sort(np.unique([int(x.with_suffix('').name.replace('history_step-', '')) for x in history_files]))
    return step_ids

def compute_Pavg(mesh):
    """Converts CPRESS to cell data and computes area weighted average pressure"""
    tet_surf = mesh.extract_cells_by_type(10).extract_surface(algorithm=None)
    P = tet_surf.point_data_to_cell_data().cell_data['CPRESS'] 
        # CPRESS comes as point data
        # this assigns to each elements CPRESS value, the average of the CPRESS of the 3 points of the element
        # Edge case: elements has point CPRESS = [0, 0, 1.5], element given CPRESS = 0.5, but entire area considered in contact for avg
        # This slightly overestimates area and therefore reduces average pressure
    A = tet_surf.compute_cell_sizes()["Area"]
    mask = P > 0

    return np.sum(P[mask] * A[mask]) / np.sum(A[mask])

def build_fe_data(inp_file, csv_dir=None):
    param_path = inp_file.parents[4]/ 'params/full_params.json'
    with open(param_path, 'r') as f:
        Fs = json.load(f)['inp']['force_steps']

    if csv_dir is None:
        csv_dir = inp_file.parent / 'resultCSVs' 

    steps = get_step_ids(csv_dir)
    frame = -1 # final frame of each step
    field_metrics = ["CPRESS", "U"]


    fe_data = {} # {F1: {'tpm':mesh1, 'mc1':mesh1}, ...} - each mesh contains all fea data

    meshes_orig = inp2pv(inp_file)
    for step in steps:
        meshes = {
            bone: mesh.copy(deep=True)
            for bone, mesh in meshes_orig.items()
        }
        #meshes = inp2pv(inp_file)
        for bone, mesh in meshes.items():
            instance = f"{bone.upper()}_INST"
            
            # Field data
            for metric in field_metrics:
                field_path = get_field_path(csv_dir, metric, step, frame, instance)
                field_df = get_field_df(field_path)
                add_field_to_mesh(mesh, field_df)

            # History data
            history_data = pd.read_csv(get_history_path(csv_dir, step))
            # A
            CAREA_data = history_data[history_data['historyOutputDescription']=='Total area in contact']
            CA = CAREA_data['value'].iloc[frame]
            RF_data = history_data[history_data['historyOutputKey']=='RF1']
            RF = RF_data['value'].iloc[frame]
            U1_data = history_data[history_data['historyOutputKey']=='U1']
            U1 = U1_data['value'].iloc[frame]

            if step == 0:
                mesh.field_data['RF'] = RF*-1
            else:
                mesh.field_data['RF'] = Fs[step-1]
            mesh.field_data['U1'] = U1
            mesh.field_data['CA'] = CA

            #P_avg = np.mean(mesh['CPRESS'][mesh['CPRESS']>0])
            #Summary data
            mesh.field_data['P_max'] = mesh['CPRESS'].max()
            mesh.field_data['P_avg'] = compute_Pavg(mesh)
            mesh.field_data['COP'] = np.array(mesh.points[np.argmax(mesh['CPRESS'])])

        fe_data[step] = meshes
    return fe_data

def get_results(fe_data, bone='tpm'):
    results = []
    for step in fe_data:
        results.append({key: val[0] if key != 'COP' else [[round(x, 2) for x in val]] for key, val in dict(fe_data[step][bone].field_data).items()})
    return pd.DataFrame(results)

In [3]:
hybrid_inp_path = Path('../../../InpPipeline/outputs/testing/inpGeom2/inpFiles/50017L/inp/35T-flexion-00/35T-flexion-00-geom.inp')
manual_inp_path = Path('../../../InpPipeline/outputs/testing/inpGeom2/inpFiles/50017L/inp/35T-flexion-00/35T-flexion-00.inp')

hybrid_csv_path = Path('../../../InpPipeline/outputs/testing/inpGeom2/aire/output/resultCSVs/35T-flexion-hybrid')
manual_csv_path = Path('../../../InpPipeline/outputs/testing/inpGeom2/aire/output/resultCSVs/35T-flexion-manual')

manual = build_fe_data(manual_inp_path, manual_csv_path)
hybrid = build_fe_data(hybrid_inp_path, hybrid_csv_path)

#### TPM

In [4]:
get_results(manual, 'tpm')

,RF,U1,CA,P_max,P_avg,COP
0,0.667116,-0.050000,6.632949,0.183876,0.074856,"[[-20.99, 0.65, 6.47]]"
1,10.000000,-0.170970,17.337196,1.316180,0.446468,"[[-20.99, 0.65, 6.47]]"
2,20.000000,-0.231863,23.032879,2.137792,0.640906,"[[-21.0, 0.45, 6.38]]"


In [5]:
get_results(hybrid, 'tpm')

,RF,U1,CA,P_max,P_avg,COP
0,0.667115,-0.050000,6.632949,0.183874,0.074856,"[[-20.99, 0.65, 6.47]]"
1,10.000000,-0.170970,17.337196,1.316179,0.446468,"[[-20.99, 0.65, 6.47]]"
2,20.000000,-0.231863,23.032879,2.137793,0.640906,"[[-21.0, 0.45, 6.38]]"


#### MC1

In [57]:
get_results(manual, 'mc1')

,RF,U1,CA,P_max,P_avg,COP
0,0.667116,-0.050000,6.632949,0.186511,0.077390,"[[-20.99, 0.62, 6.49]]"
1,10.000000,-0.170970,17.337196,1.343788,0.461825,"[[-20.99, 0.62, 6.49]]"
2,20.000000,-0.231863,23.032879,2.183511,0.671877,"[[-20.95, 0.56, 6.26]]"


In [58]:
get_results(hybrid, 'mc1')

,RF,U1,CA,P_max,P_avg,COP
0,0.667115,-0.050000,6.632949,0.186507,0.077390,"[[-20.99, 0.62, 6.49]]"
1,10.000000,-0.170970,17.337196,1.343784,0.461825,"[[-20.99, 0.62, 6.49]]"
2,20.000000,-0.231863,23.032879,2.183521,0.671877,"[[-20.95, 0.56, 6.26]]"


#### full_run results

In [5]:
inp_path = Path('../../../InpPipeline/outputs/testing/inpGeom3/inpFiles/50017L/inp/35T-flexion-00/35T-flexion-00-geom.inp')
csv_path = Path('../../../InpPipeline/outputs/testing/inpGeom3/aire/output/resultCSVs/50017L-35T-flexion-00')

full = build_fe_data(inp_path, csv_path)

In [7]:
get_results(full, 'mc1')

,RF,U1,CA,P_max,P_avg,COP
0,0.667116,-0.050000,6.632949,0.186511,0.077390,"[[-20.99, 0.62, 6.49]]"
1,10.000000,-0.170970,17.337196,1.343788,0.461825,"[[-20.99, 0.62, 6.49]]"
2,20.000000,-0.231863,23.032877,2.183511,0.671877,"[[-20.95, 0.56, 6.26]]"
3,50.000000,-0.338328,36.692989,4.031354,0.915302,"[[-20.9, 0.79, 6.18]]"
